# Closed Loop Geothermal Supply (Plymouth)

## 1. Imports

In [1]:

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from pathlib import Path
import pyogrio

## 2. Project and output paths


In [2]:
PROJECT_DIR = next(candidate 
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (candidate / "01_Data").exists())

In [3]:
OUTPUT_DIR = PROJECT_DIR / "01_Data/Processed/Geothermal/model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREPARED_GEOLOGY_GPKG = OUTPUT_DIR / "01_prepared_geology.gpkg"
BOUNDARY_GPKG = PROJECT_DIR / "01_Data/Processed/Boundaries/plymouth_boundaries.gpkg"


TARGET_CRS = "EPSG:27700"
print("Prepared geology file exists:", PREPARED_GEOLOGY_GPKG.exists())

Prepared geology file exists: True


## 3. Load prepared geology


In [4]:
#load plymoouth geology data
plymouth_bedrock = gpd.read_file(PREPARED_GEOLOGY_GPKG, layer="prepared_geology")

print("Plymouth geology shape:", plymouth_bedrock.shape)
print("Distinct lithologies:  ", plymouth_bedrock["RCS_D"].nunique())
print("Geology Unit - lithology combinations: ", plymouth_bedrock["model_unit_lithology_id"].nunique())
print("CRS:", plymouth_bedrock.crs)

plymouth_bedrock[["LEX_D", "RCS_D", "model_unit_lithology_id", "area_km2", "thermal_conductivity_w_mk"]].head(10)

Plymouth geology shape: (341, 49)
Distinct lithologies:   16
Geology Unit - lithology combinations:  32
CRS: EPSG:27700


,LEX_D,RCS_D,model_unit_lithology_id,area_km2,thermal_conductivity_w_mk
0,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.068762,3.0
1,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.042912,3.0
2,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.042779,3.0
3,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.024289,3.0
4,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.022360,3.0
5,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.019742,3.0
6,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.011782,3.0
7,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.002804,3.0
8,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.000419,3.0
9,FARADAY ROAD MEMBER,LIMESTONE,FAR__LIMESTONE,0.000216,3.0


In [5]:
# Check for missing values in required fields
required_fields = ["LEX", "LEX_D", "RCS_D", "model_unit_lithology_id", 
                   "area_m2", "area_km2","thermal_conductivity_w_mk", "lambda_low_w_mk", 
                   "lambda_high_w_mk","volumetric_heat_capacity_mj_m3k"
]
plymouth_bedrock[required_fields].isnull().sum()

LEX                                0
LEX_D                              0
RCS_D                              0
model_unit_lithology_id            0
area_m2                            0
area_km2                           0
thermal_conductivity_w_mk          0
lambda_low_w_mk                    0
lambda_high_w_mk                   0
volumetric_heat_capacity_mj_m3k    0
dtype: int64

## 5. Define model assumptions


In [6]:
# create a dataframe of model assumptions
MODEL_ASSUMPTIONS = {
    "ground_temperature_c":             14.05,                 # T0 (deg C)
    "borehole_depth_m":                 150.0,                 # L (m)
    "borehole_radius_m":                0.075,                 # rb (m)
    "borehole_thermal_resistance_mk_w": 0.068,                 # Rb (m K/W)
    "min_fluid_temperature_c":          -2,                  # Tlim – heating mode minimum (deg C)
    "heating_season_days":              182.0,                 # tc – seasonal duration (days)
    "simulation_lifetime_years":        50.0,                  # ts – long-term simulation period
    "borehole_spacing_m":               75.0,                  # uniform grid spacing 
    "heat_pump_scop":                    3.2,                   # for useful heat conversion
}

assumptions = pd.DataFrame([{"parameter": k, "value": v} for k, v in MODEL_ASSUMPTIONS.items()])
assumptions

,parameter,value
0,ground_temperature_c,14.050
1,borehole_depth_m,150.000
2,borehole_radius_m,0.075
3,borehole_thermal_resistance_mk_w,0.068
4,min_fluid_temperature_c,-2.000
5,heating_season_days,182.000
6,simulation_lifetime_years,50.000
7,borehole_spacing_m,75.000
8,heat_pump_scop,3.200


## 6. Load Plymouth boundary


In [7]:
# Load the same clipped LSOA boundary used by the demand workflow

plymouth_lsoa = gpd.read_file(BOUNDARY_GPKG,layer="lsoa_plymouth_2021_clipped").to_crs(TARGET_CRS)

plymouth_lsoa = (plymouth_lsoa[plymouth_lsoa.geometry.notna()].copy())

plymouth_lsoa["geometry"] = (plymouth_lsoa.geometry.make_valid())

# Dissolve individual LSOAs into one boundary
plymouth_boundary = gpd.GeoDataFrame(geometry=[plymouth_lsoa.geometry.union_all()],crs=TARGET_CRS)

plymouth_boundary["geometry"] = (plymouth_boundary.geometry.make_valid())

print("boundary CRS:", plymouth_boundary.crs)

print("boundary area (km²):",plymouth_boundary.geometry.area.sum()/ 1_000_000
)

boundary CRS: EPSG:27700
boundary area (km²): 79.7954836012946


## 7. Prepare geology for city-wide supply assessment


In [8]:
# Check how much of the Plymouth study area is covered by mapped water

OS_OPENMAP_DIR = PROJECT_DIR / "01_Data/Raw/OS_OpenMapLocal_Digimap/Download_Plymouth_OS_OpenMapLocal_Digimap_2999520/open-map-local_6431409"

water_layers = {"Surface water": OS_OPENMAP_DIR / "SX_SurfaceWater_Area.shp",
                "Tidal water": OS_OPENMAP_DIR / "SX_TidalWater.shp",
                "Foreshore": OS_OPENMAP_DIR / "SX_Foreshore.shp"
                }

water_geometries = []

for name, path in water_layers.items():

    water = gpd.read_file(path).to_crs(TARGET_CRS)
    water = water[water.geometry.notna()].copy()
    water["geometry"] = water.geometry.make_valid()

    water = gpd.clip(water, plymouth_boundary)
    water = water[water.geometry.notna() & ~water.geometry.is_empty].copy()

    area_km2 = water.geometry.area.sum() / 1_000_000

    print(f"{name} area within Plymouth (km2):", round(area_km2, 4))

    if len(water):
        water_geometries.append(water.geometry.union_all())



Surface water area within Plymouth (km2): 0.5694
Tidal water area within Plymouth (km2): 0.1858
Foreshore area within Plymouth (km2): 0.1562


In [9]:
# Combine the water layers so overlapping areas are not counted twice
if water_geometries:

    water_union = gpd.GeoSeries(water_geometries,crs=TARGET_CRS).union_all()

    water_area_km2 = water_union.area / 1_000_000

else:
    water_area_km2 = 0

plymouth_area_km2 = plymouth_boundary.geometry.area.sum() / 1_000_000
land_area_km2 = plymouth_area_km2 - water_area_km2


print("Plymouth boundary area (km2):", round(plymouth_area_km2, 4))
print("Unique mapped water area (km2):", round(water_area_km2, 4))
print("Area excluding mapped water (km2):", round(land_area_km2, 4))
print("Water share of study area (%):", round((water_area_km2 / plymouth_area_km2) * 100, 2))

Plymouth boundary area (km2): 79.7955
Unique mapped water area (km2): 0.7552
Area excluding mapped water (km2): 79.0403
Water share of study area (%): 0.95


In [10]:
# Use the full geology area, excluding surface and tidal water
closed_loop = plymouth_bedrock.copy()
closed_loop["geology_polygon_id"] = np.arange(len(closed_loop))

# Remove water area from the geology polygons
closed_loop["geometry"] = closed_loop.geometry.apply(lambda geom: geom.difference(water_union))

closed_loop = closed_loop[~closed_loop.geometry.isna() & ~closed_loop.geometry.is_empty].copy()

closed_loop["model_area_m2"] = closed_loop.geometry.area
closed_loop["model_area_km2"] = closed_loop["model_area_m2"] / 1_000_000

print("Total mapped geology area excluding water (km2):",round(closed_loop["model_area_km2"].sum(), 2))

closed_loop[["LEX_D","RCS_D","area_km2","model_area_km2"]].head()

Total mapped geology area excluding water (km2): 79.04


,LEX_D,RCS_D,area_km2,model_area_km2
0,FARADAY ROAD MEMBER,LIMESTONE,0.068762,0.068762
1,FARADAY ROAD MEMBER,LIMESTONE,0.042912,0.038222
2,FARADAY ROAD MEMBER,LIMESTONE,0.042779,0.042273
3,FARADAY ROAD MEMBER,LIMESTONE,0.024289,0.024289
4,FARADAY ROAD MEMBER,LIMESTONE,0.022360,0.022063


## 8. G.POT per borehole energy calculation


In [11]:
# G.POT by Casasso and Sethi (2016)
# Calculates annual average ground thermal power per borehole for each geology polygon

ONE_YEAR_SECONDS       = 365.25 * 24 * 3600
HEATING_SEASON_SECONDS = MODEL_ASSUMPTIONS["heating_season_days"] * 24 * 3600
SIMULATION_SECONDS     = MODEL_ASSUMPTIONS["simulation_lifetime_years"] * ONE_YEAR_SECONDS
tc_ratio               = HEATING_SEASON_SECONDS / ONE_YEAR_SECONDS  # fraction of year used for heating
T0   = MODEL_ASSUMPTIONS["ground_temperature_c"]
Tlim = MODEL_ASSUMPTIONS["min_fluid_temperature_c"]
L    = MODEL_ASSUMPTIONS["borehole_depth_m"]
rb   = MODEL_ASSUMPTIONS["borehole_radius_m"]
Rb   = MODEL_ASSUMPTIONS["borehole_thermal_resistance_mk_w"]

#conductivity 
conductivity_values = [
    ("low", "lambda_low_w_mk"),
    ("representative", "thermal_conductivity_w_mk"),
    ("high","lambda_high_w_mk"),
]

In [12]:
#model equations 
for scenario, lambda_col in conductivity_values :
    lam   = closed_loop[lambda_col]
    Cv    = closed_loop["volumetric_heat_capacity_mj_m3k"] * 1_000_000  # convert to (J/m³K)
    alpha = lam / Cv                                                    # thermal diffusivity (m²/s)

    u_s = rb**2 / (4 * alpha * SIMULATION_SECONDS)       # dimensionless time at simulation end
    u_c = rb**2 / (4 * alpha * HEATING_SEASON_SECONDS)   # dimensionless time at end of heating season

    # G.POT denominator (log approximation to the infinite line-source integral)
    denominator = (
        -0.619 * tc_ratio * np.log(u_s)
        + (0.532 * tc_ratio - 0.962) * np.log(u_c)
        - 0.455 * tc_ratio
        - 1.619
        + 4 * np.pi * lam * Rb
    )

    # Annual-average ground thermal power per borehole (W)
    q_bhe_w = 8 * (T0 - Tlim) * lam * L * tc_ratio / denominator

    closed_loop[f"{scenario}_gpot_w_per_borehole"] = q_bhe_w
    closed_loop[f"{scenario}_gpot_w_per_m"]        = q_bhe_w / L
    closed_loop[f"{scenario}_ground_energy_mwh_per_borehole_year"] = q_bhe_w * 8760 / 1_000_000

closed_loop[[
    "model_unit_lithology_id",
    "thermal_conductivity_w_mk",
    "representative_gpot_w_per_m",
    "representative_ground_energy_mwh_per_borehole_year",
]].head(10)

,model_unit_lithology_id,thermal_conductivity_w_mk,representative_gpot_w_per_m,representative_ground_energy_mwh_per_borehole_year
0,FAR__LIMESTONE,3.0,16.346675,21.479531
1,FAR__LIMESTONE,3.0,16.346675,21.479531
2,FAR__LIMESTONE,3.0,16.346675,21.479531
3,FAR__LIMESTONE,3.0,16.346675,21.479531
4,FAR__LIMESTONE,3.0,16.346675,21.479531
5,FAR__LIMESTONE,3.0,16.346675,21.479531
6,FAR__LIMESTONE,3.0,16.346675,21.479531
7,FAR__LIMESTONE,3.0,16.346675,21.479531
8,FAR__LIMESTONE,3.0,16.346675,21.479531
9,FAR__LIMESTONE,3.0,16.346675,21.479531


## 9. Calculate geothermal supply


### Generate uniform BHE grid and calculate supply


In [13]:
#set model assumptions for the grid calculation
SCOP = MODEL_ASSUMPTIONS["heat_pump_scop"]
spacing = MODEL_ASSUMPTIONS["borehole_spacing_m"]

#select geology and thermal fields for each BHE point
geology_fields_for_grid = [
    "geology_polygon_id",
    "LEX_D",
    "RCS_D",
    "model_unit_lithology_id",
    "thermal_conductivity_w_mk",
    "lambda_low_w_mk",
    "lambda_high_w_mk",
    "volumetric_heat_capacity_mj_m3k",

    "low_gpot_w_per_borehole",
    "representative_gpot_w_per_borehole",
    "high_gpot_w_per_borehole",

    "low_gpot_w_per_m",
    "representative_gpot_w_per_m",
    "high_gpot_w_per_m",

    "low_ground_energy_mwh_per_borehole_year",
    "representative_ground_energy_mwh_per_borehole_year",
    "high_ground_energy_mwh_per_borehole_year",

    "geometry",
]


In [14]:
#function to build a uniform BHE grid
def build_uniform_bhe_grid(spacing_m, full_attributes=True):
    """
    Generate a square BHE grid at the specified spacing,
    retain points falling within the terrestrial geology,
    and assign the geology beneath each point.
    """

    minx, miny, maxx, maxy = closed_loop.total_bounds

    #anchor grid-cell centres to British National Grid coordinates
    x_start = (np.floor(minx / spacing_m) * spacing_m + spacing_m / 2)

    y_start = (np.floor(miny / spacing_m) * spacing_m+ spacing_m / 2)

    x_coords = np.arange(x_start,maxx + spacing_m,spacing_m)

    y_coords = np.arange(y_start,maxy + spacing_m,spacing_m)

    xx, yy = np.meshgrid(x_coords, y_coords)

    candidate_grid = gpd.GeoDataFrame({"grid_id": np.arange(xx.size)},
        geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()), crs=closed_loop.crs)

    #retain full geological information for the main 75 m grid
    if full_attributes:
        join_fields = geology_fields_for_grid
    else:
        join_fields = ["representative_ground_energy_mwh_per_borehole_year", "geometry",]

    grid = gpd.sjoin(candidate_grid, closed_loop[join_fields], how="inner",predicate="within")

    grid = (grid.drop(columns=["index_right"]).reset_index(drop=True))

    #each grid point should match only one geology polygon
    if grid["grid_id"].duplicated().any():
        raise ValueError(
            "Duplicate BHE grid points detected after geological assignment."
        )

    grid["bhe_id"] = np.arange(1,len(grid) + 1)

    grid["grid_spacing_m"] = spacing_m
    grid["easting_m"] = grid.geometry.x
    grid["northing_m"] = grid.geometry.y

    return grid


In [15]:
#create baseline 75 m BHE grid
bhe_grid = build_uniform_bhe_grid( spacing_m=spacing,full_attributes=True)

print("Uniform-grid BHE points retained:", len(bhe_grid))

Uniform-grid BHE points retained: 14056


In [16]:
#calculate annual energy for every actual BHE
for thermal_scenario in ["low","representative","high"]:

    bhe_grid[f"{thermal_scenario}_ground_energy_mwh_year"] = bhe_grid[
        f"{thermal_scenario}_ground_energy_mwh_per_borehole_year"]

    bhe_grid[f"{thermal_scenario}_useful_heat_mwh_year"] = (bhe_grid[
        f"{thermal_scenario}_ground_energy_mwh_year"
        ]* SCOP/ (SCOP - 1)
    )


In [17]:
#aggregate BHE point results back to geology polygons
grid_polygon_summary = (
    bhe_grid
    .groupby(
        "geology_polygon_id",
        as_index=False
    )
    .agg(
        borehole_count=(
            "bhe_id",
            "size"
        ),

        low_total_ground_energy_mwh_year=(
            "low_ground_energy_mwh_year",
            "sum"
        ),

        representative_total_ground_energy_mwh_year=(
            "representative_ground_energy_mwh_year",
            "sum"
        ),

        high_total_ground_energy_mwh_year=(
            "high_ground_energy_mwh_year",
            "sum"
        ),

        low_useful_heat_mwh_year=(
            "low_useful_heat_mwh_year",
            "sum"
        ),

        representative_useful_heat_mwh_year=(
            "representative_useful_heat_mwh_year",
            "sum"
        ),

        high_useful_heat_mwh_year=(
            "high_useful_heat_mwh_year",
            "sum"
        ),
    )
)


In [18]:
#remove old area-based supply fields if they already exist
legacy_supply_fields = [
    "equivalent_borehole_count",
    "borehole_count",

    "low_total_ground_energy_mwh_year",
    "representative_total_ground_energy_mwh_year",
    "high_total_ground_energy_mwh_year",

    "low_useful_heat_mwh_year",
    "representative_useful_heat_mwh_year",
    "high_useful_heat_mwh_year",
]

closed_loop = closed_loop.drop(
    columns=[
        field
        for field in legacy_supply_fields
        if field in closed_loop.columns
    ]
)

#join grid-derived totals to the geology polygons
closed_loop = closed_loop.merge(
    grid_polygon_summary,
    on="geology_polygon_id",
    how="left"
)


In [19]:
#fill polygons with no 75 m grid point as zero supply
grid_supply_fields = [
    "borehole_count",

    "low_total_ground_energy_mwh_year",
    "representative_total_ground_energy_mwh_year",
    "high_total_ground_energy_mwh_year",

    "low_useful_heat_mwh_year",
    "representative_useful_heat_mwh_year",
    "high_useful_heat_mwh_year",
]

closed_loop[grid_supply_fields] = (
    closed_loop[grid_supply_fields]
    .fillna(0)
)

closed_loop["borehole_count"] = (
    closed_loop["borehole_count"]
    .astype(int)
)


In [20]:
#summarise  geothermal supply results
print("Total actual BHEs:",len(bhe_grid))

print("Representative ground energy (GWh/year):",round(bhe_grid["representative_ground_energy_mwh_year"].sum() / 1000,2))

print("Representative useful heat (GWh/year):",round(bhe_grid["representative_useful_heat_mwh_year"].sum() / 1000,2))

closed_loop[[ "model_unit_lithology_id","model_area_km2","borehole_count","representative_total_ground_energy_mwh_year",
        "representative_useful_heat_mwh_year",]].head(20)


Total actual BHEs: 14056
Representative ground energy (GWh/year): 273.91
Representative useful heat (GWh/year): 398.41


,model_unit_lithology_id,model_area_km2,borehole_count,representative_total_ground_energy_mwh_year,representative_useful_heat_mwh_year
0,FAR__LIMESTONE,6.876150e-02,13,279.233905,406.158407
1,FAR__LIMESTONE,3.822196e-02,9,193.315780,281.186589
2,FAR__LIMESTONE,4.227310e-02,9,193.315780,281.186589
3,FAR__LIMESTONE,2.428900e-02,4,85.918124,124.971817
4,FAR__LIMESTONE,2.206286e-02,3,64.438593,93.728863
5,FAR__LIMESTONE,1.974200e-02,3,64.438593,93.728863
6,FAR__LIMESTONE,1.157088e-02,2,42.959062,62.485909
7,FAR__LIMESTONE,2.804000e-03,0,0.000000,0.000000
8,FAR__LIMESTONE,4.134231e-04,0,0.000000,0.000000
9,FAR__LIMESTONE,2.139918e-04,0,0.000000,0.000000


## 10. Summarise by geological unit


In [21]:
#summarise geothermal supply by geological unit
summary_keys = ["LEX_D", "RCS_D", "model_unit_lithology_id"]


In [22]:
#summarise geology area by unit
geology_summary = (closed_loop.groupby( summary_keys,
        dropna=False).agg(
        polygon_count=("geometry","size"),
        area_km2=("area_km2","sum"),
        model_area_km2=("model_area_km2","sum"),
        mean_gpot_w_per_m=( "representative_gpot_w_per_m","mean"),).reset_index())

In [23]:
#summarise BHE counts and supply by unit
grid_supply_summary = (bhe_grid.groupby(summary_keys,
        dropna=False).agg(
        borehole_count=("bhe_id","size"),
        low_total_ground_energy_mwh_year=("low_ground_energy_mwh_year","sum"),
        representative_total_ground_energy_mwh_year=("representative_ground_energy_mwh_year","sum"),
        high_total_ground_energy_mwh_year=("high_ground_energy_mwh_year","sum"),
        low_useful_heat_mwh_year=("low_useful_heat_mwh_year","sum"),
        representative_useful_heat_mwh_year=("representative_useful_heat_mwh_year","sum"),
        high_useful_heat_mwh_year=("high_useful_heat_mwh_year","sum"),).reset_index())

In [24]:
#combine geology information with grid supply
supply_summary = geology_summary.merge(grid_supply_summary,on=summary_keys,how="left")


In [25]:
#fill geological units with no 75 m grid points
zero_fill_fields = [
    "borehole_count",
    "low_total_ground_energy_mwh_year",
    "representative_total_ground_energy_mwh_year",
    "high_total_ground_energy_mwh_year",
    "low_useful_heat_mwh_year",
    "representative_useful_heat_mwh_year",
    "high_useful_heat_mwh_year",
]

supply_summary[zero_fill_fields] = (supply_summary[zero_fill_fields].fillna(0))

supply_summary["borehole_count"] = (supply_summary["borehole_count"].astype(int))


In [26]:
#sort by representative geothermal contribution
supply_summary = (supply_summary.sort_values("representative_total_ground_energy_mwh_year",ascending=False).reset_index(drop=True))
supply_summary


,LEX_D,RCS_D,model_unit_lithology_id,polygon_count,area_km2,model_area_km2,mean_gpot_w_per_m,borehole_count,low_total_ground_energy_mwh_year,representative_total_ground_energy_mwh_year,high_total_ground_energy_mwh_year,low_useful_heat_mwh_year,representative_useful_heat_mwh_year,high_useful_heat_mwh_year
0,UPPER DEVONIAN SLATES,SLATE,UDVS__SLATE,3,25.880615,25.732843,15.140484,4585,78911.072949,91216.723069,96179.989233,114779.742471,132678.869918,139898.166158
1,SALTASH FORMATION,SLATE AND SILTSTONE,SAH__SLATE AND SILTSTONE,68,17.934718,17.684625,14.303482,3142,53979.836929,59053.182177,62403.691757,78516.126442,85895.537713,90769.006192
2,TORPOINT FORMATION,MUDSTONE AND SILTSTONE,TPT__MUDSTONE AND SILTSTONE,52,11.636274,11.548139,14.397990,2048,34135.538181,38746.026674,42938.988835,49651.691900,56357.856980,62456.711032
3,MIDDLE DEVONIAN SLATES,SLATE,MDVS__SLATE,7,7.554552,7.543210,15.140484,1345,23148.395445,26758.231740,28214.195315,33670.393375,38921.064349,41038.829549
4,MIDDLE DEVONIAN LIMESTONE,LIMESTONE,MDVL__LIMESTONE,7,5.249781,5.088488,16.346675,896,17872.523603,19245.659885,20124.586140,25996.397968,27993.687105,29272.125295
5,TAVY FORMATION,SLATE,TVY__SLATE,1,3.337381,3.324004,15.140484,599,10309.211057,11916.863057,12565.281036,14995.216083,17333.618993,18276.772416
6,PLYMOUTH LIMESTONE FORMATION,LIMESTONE,PYL__LIMESTONE,49,1.457051,1.424127,16.346675,254,5066.541289,5455.800905,5704.960803,7369.514603,7935.710407,8298.124805
7,MIDDLE DEVONIAN SLATES,"PYROCLASTIC-ROCK, BASALTIC","MDVS__PYROCLASTIC-ROCK, BASALTIC",7,1.219067,1.215461,13.064702,223,3298.619169,3828.245011,4200.234038,4797.991519,5568.356379,6109.431328
8,WEARDE SANDSTONE MEMBER,"SANDSTONE, SILTSTONE AND MUDSTONE","WRG__SANDSTONE, SILTSTONE AND MUDSTONE",2,0.877768,0.877702,15.571096,157,2775.468102,3212.285912,3432.174623,4037.044512,4672.415872,4992.253998
9,UPPER DEVONIAN SLATES,"PYROCLASTIC-ROCK, BASALTIC","UDVS__PYROCLASTIC-ROCK, BASALTIC",15,1.016773,1.015390,13.064702,180,2662.562558,3090.063237,3390.323439,3872.818266,4494.637436,4931.379547


In [27]:
#summarise city-wide geothermal supply from the BHE grid
for scenario in ["low", "representative","high"]:
    ground_gwh = (bhe_grid[f"{scenario}_ground_energy_mwh_year"].sum()/ 1000)
    useful_heat_gwh = (bhe_grid[f"{scenario}_useful_heat_mwh_year"].sum()/ 1000)

    print(scenario,"| BHE count:",len(bhe_grid),"| Ground energy (GWh/year):",round(ground_gwh, 2),
          "| Useful heat (GWh/year):", round(useful_heat_gwh, 2))


low | BHE count: 14056 | Ground energy (GWh/year): 242.46 | Useful heat (GWh/year): 352.67
representative | BHE count: 14056 | Ground energy (GWh/year): 273.91 | Useful heat (GWh/year): 398.41
high | BHE count: 14056 | Ground energy (GWh/year): 291.58 | Useful heat (GWh/year): 424.12


## 11. Save outputs


In [28]:
#define output files
output_gpkg = (OUTPUT_DIR/ "02_closed_loop_supply.gpkg")

summary_csv = (OUTPUT_DIR/ "02_closed_loop_supply_summary.csv")

assumptions_csv = (OUTPUT_DIR/ "02_closed_loop_model_assumptions.csv")


In [29]:
#recreate the GeoPackage
if output_gpkg.exists():
    output_gpkg.unlink()

In [30]:
#save geology polygons and the BHE grid
closed_loop.to_file(output_gpkg,layer="closed_loop_supply",driver="GPKG")

bhe_grid.to_file(output_gpkg,layer="bhe_grid_75m",driver="GPKG",mode="a")


In [31]:
#save tabular outputs
supply_summary.to_csv(summary_csv,index=False)

assumptions.to_csv(assumptions_csv,index=False)


In [ ]:
#export thermal properties table
TABLE_OUTPUT_DIR = PROJECT_DIR / "03_Outputs/Tables"
TABLE_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

thermal_properties_csv = PROJECT_DIR / "01_Data/Processed/Geothermal/Plymouth_Thermal_Properties_Final.csv"
thermal_properties = pd.read_csv(thermal_properties_csv,encoding="utf-8-sig").copy()

thermal_properties_table = supply_summary.merge(
    thermal_properties[["mapped_lithology","thermal_conductivity_w_mk","lambda_low_w_mk","lambda_high_w_mk",
                        "density_kg_m3","specific_heat_capacity_j_kgk","volumetric_heat_capacity_mj_m3k","primary_source"]],
    left_on="RCS_D",right_on="mapped_lithology",how="left",validate="many_to_one"
)
if thermal_properties_table["thermal_conductivity_w_mk"].isna().any():
    raise ValueError("Some unit-lithology rows do not have thermal-property values.")

thermal_properties_table["Unit code"] = thermal_properties_table["model_unit_lithology_id"].str.split("__").str[0]
thermal_properties_table["Geological unit"] = thermal_properties_table["LEX_D"].str.title().str.replace(" And "," and ",regex=False)
thermal_properties_table["Mapped lithology"] = (thermal_properties_table["RCS_D"].str.title().str.replace(" And "," and ",regex=False)
                                         .str.replace("Pyroclastic-Rock","Pyroclastic Rock",regex=False)
                                         .str.replace("Basaltic-Rock","Basaltic Rock",regex=False)
                                         .str.replace("Subequal/Subordinate","subequal/Subordinate",regex=False))
thermal_properties_table["Model area (km2)"] = thermal_properties_table["model_area_km2"].round(2)
thermal_properties_table["Share of model area (%)"] = (thermal_properties_table["model_area_km2"] / thermal_properties_table["model_area_km2"].sum() * 100).round(1)
thermal_properties_table["Low thermal conductivity (W m-1 K-1)"] = thermal_properties_table["lambda_low_w_mk"].round(2)
thermal_properties_table["Representative thermal conductivity (W m-1 K-1)"] = thermal_properties_table["thermal_conductivity_w_mk"].round(2)
thermal_properties_table["High thermal conductivity (W m-1 K-1)"] = thermal_properties_table["lambda_high_w_mk"].round(2)
thermal_properties_table["Density (kg m-3)"] = thermal_properties_table["density_kg_m3"].round(0).astype(int)
thermal_properties_table["Specific heat capacity (J kg-1 K-1)"] = thermal_properties_table["specific_heat_capacity_j_kgk"].round(0).astype(int)
thermal_properties_table["Volumetric heat capacity (J m-3 K-1)"] = (thermal_properties_table["volumetric_heat_capacity_mj_m3k"] * 1_000_000).round(0).astype(int)
thermal_properties_table["Source"] = thermal_properties_table["primary_source"]

thermal_properties_table = thermal_properties_table[["Unit code","Geological unit","Mapped lithology","Model area (km2)","Share of model area (%)",
                                     "Low thermal conductivity (W m-1 K-1)","Representative thermal conductivity (W m-1 K-1)",
                                     "High thermal conductivity (W m-1 K-1)","Density (kg m-3)","Specific heat capacity (J kg-1 K-1)",
                                     "Volumetric heat capacity (J m-3 K-1)","Source"]]
thermal_properties_table.to_csv(TABLE_OUTPUT_DIR / "Geological_Thermal_Properties.csv",index=False)